# 05 · Acto 3: el LLM hace lo que sí sabe (explicar el estilo con palabras)

En los notebooks 02–04 vimos que un LLM pequeño **no** es buena herramienta para *leer números* de color de una foto: una regresión de 20 líneas gana. Pero un LLM es, ante todo, un modelo de **lenguaje**. Así que le damos la tarea al revés:

> dado el JSON de ajustes (lo produzca quien lo produzca: C_color, una persona…), escribir **una explicación corta en español** para el usuario.

Ejemplo: `{"exposicion": -0.38, "temperatura": 0.66, ...}` → «La referencia es más oscura y más cálida: se baja bastante la exposición y se sube bastante la temperatura.»

**Datos fabricados otra vez.** Las frases objetivo salen de **reglas** a partir de cada JSON (`comun_estilo.frase_objetivo`): se ignoran los ajustes casi nulos, la intensidad decide «un poco» o «bastante», el orden de los ajustes es fijo y hay **3 plantillas** de frase para que el modelo no se limite a copiar una sola. Como las frases salen de reglas, también podemos **medir automáticamente** si una frase es correcta.

Usamos solo la **parte de lenguaje** de SmolVLM-256M-Instruct (sin imagen; licencia Apache-2.0), para comparar con las versiones anteriores. Entrada: `parejas/parejas.jsonl` (solo los JSON de ajustes). Salida: `resultados/metricas_explicacion.json`, `resultados/explicaciones.jsonl`, `adaptador_explicacion/` y dos runs en MLflow.

In [1]:
import os
N_EJEMPLOS = int(os.environ.get("ESTILO_N_EJEMPLOS", "60"))   # parejas del 01: de ahí salen val y test
EPOCAS = int(os.environ.get("ESTILO_EPOCAS", "2"))
RESOLUCION = int(os.environ.get("ESTILO_RESOLUCION", "224"))  # no se usa aquí (no hay imágenes)
MODELO = os.environ.get("ESTILO_MODELO", "HuggingFaceTB/SmolVLM-256M-Instruct")
CARPETA_TRABAJO = os.environ.get("ESTILO_TRABAJO", "_trabajo")
# Train de B3: JSON fabricados al azar (sin fotos, gratis). Clase: 500 y 2 épocas. CI: 100 y 1 época (sin verificar).
N_FRASES = int(os.environ.get("ESTILO_N_FRASES", "500"))
# «facil»: sentido e intensidad ya calculados en la entrada (el modelo solo redacta); «json»: el JSON tal cual
FORMATO = os.environ.get("ESTILO_FORMATO", "facil")
TAM_LOTE = 4
TASA_APRENDIZAJE = 5e-4
RANGO_LORA = 16

In [2]:
import json, random
import numpy as np
import matplotlib.pyplot as plt
import comun_estilo as ce

plt.rcParams["figure.dpi"] = 55
trabajo = ce.carpeta_trabajo(CARPETA_TRABAJO)
resultados = trabajo / "resultados"; resultados.mkdir(exist_ok=True)
hilos = ce.limitar_hilos()
import torch, transformers, peft
from peft import LoraConfig, get_peft_model
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| hilos:", hilos)
torch.manual_seed(0); random.seed(0)
proc, vlm = ce.cargar_vlm(MODELO)
tok = proc.tokenizer

# Train: N_FRASES juegos de ajustes fabricados al azar (como en el 01, pero sin fotos: la tarea es solo texto).
# Val y test: los JSON de las parejas del notebook 01 (el test es el mismo que en el 04).
rng = np.random.default_rng(4)
train = [{"id": f"gen_{i}", "ajustes": ce.muestrear_ajustes(rng)} for i in range(N_FRASES)]
val, test = ce.leer_parejas(trabajo, "val"), ce.leer_parejas(trabajo, "test")
for f in train + val + test:
    f["frase"] = ce.frase_objetivo(f["ajustes"], f["id"])
print(len(train), "train (fabricados),", len(val), "val,", len(test), "test | formato de entrada:", FORMATO)

torch 2.15.0+cpu | transformers 5.23.0 | peft 0.22.0 | hilos: 12


500 train (fabricados), 30 val, 60 test | formato de entrada: facil


## 1. Las frases objetivo y cómo se miden

Un ajuste es **relevante** si su intensidad (distancia al valor neutro dividida por la mitad de su rango) es al menos 0,15. Para cada frase medimos, con expresiones regulares:

- **% de ajustes en el sentido correcto**: de los ajustes relevantes, cuántos menciona la frase en la dirección buena («se sube la temperatura» o «más cálida» si la temperatura es positiva).
- **% en sentido contrario**: los menciona, pero al revés (el error más grave para el usuario).
- **% de frases con alucinación**: frases que hablan de un ajuste que **no** es relevante (que no se tocó).
- **% en español y de longitud razonable** (20–300 caracteres; sin palabras típicas del inglés).
- **% de frases perfectas**: todos los ajustes bien y ninguno inventado.

Las propias frases objetivo sacan 100 % en todo (lo comprobamos abajo): son el techo.

In [3]:
for f in train[:3]:
    print(ce.a_json(f["ajustes"]), "\n  ->", f["frase"], "\n")
print("Frases objetivo de test:", ce.resumir_frases([ce.evaluar_frase(f["frase"], f["ajustes"]) for f in test]))

def ids_prompt(texto_usuario):
    mensajes = [{"role": "user", "content": [{"type": "text", "text": texto_usuario}]}]
    return tok(proc.apply_chat_template(mensajes, add_generation_prompt=True), return_tensors="pt")["input_ids"][0]

def explicar(modelo, texto_usuario, max_tokens=110):
    ids = ids_prompt(texto_usuario)[None]
    with torch.no_grad():
        out = modelo.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

def medir(modelo, prompt_de, nombre):
    salidas, tiempos = [], []
    for f in test:
        with ce.Cronometro() as c:
            salidas.append(explicar(modelo, prompt_de(f)))
        tiempos.append(c.s)
    m = ce.resumir_frases([ce.evaluar_frase(s, f["ajustes"]) for s, f in zip(salidas, test)])
    m["s_por_frase"] = float(np.mean(tiempos))
    print(nombre, {k: round(v, 1) if k.startswith("pct") else round(v, 2) for k, v in m.items()})
    return salidas, m

{"exposicion": 0.62, "temperatura": 0.02, "tinte": 0.95, "contraste": 0.67, "saturacion": 1.17} 
  -> La referencia es más luminosa, más magenta, con menos contraste y más saturada: se sube bastante la exposición, se lleva el tinte bastante hacia el magenta, se baja bastante el contraste y se sube un poco la saturación. 

{"exposicion": -0.17, "temperatura": 0.6, "tinte": -0.65, "contraste": 1.38, "saturacion": 1.07} 
  -> La referencia es más oscura, más cálida, más verdosa y con más contraste: se baja un poco la exposición, se sube bastante la temperatura, se lleva el tinte bastante hacia el verde y se sube bastante el contraste. 

{"exposicion": 0.56, "temperatura": -0.05, "tinte": -0.14, "contraste": 1.31, "saturacion": 1.77} 
  -> La referencia es más luminosa, con más contraste y más saturada: se sube bastante la exposición, se sube bastante el contraste y se sube bastante la saturación. 

Frases objetivo de test: {'pct_ajustes_sentido_correcto': 100.0, 'pct_ajustes_sentido_contr

## 2. A3 · Zero-shot con un ejemplo (*one-shot*)

Le explicamos la tarea y le damos **un** ejemplo resuelto. No aprende nada: solo imita lo que ve en el prompt.

In [4]:
ejemplo = train[0]
def prompt_A3(f):
    return ("Explica en español, en una o dos frases, qué cambios de color hay que hacer. "
            "Menciona solo los ajustes que cambian.\n"
            f"Ejemplo:\n{ce.entrada_explicacion(ejemplo['ajustes'], FORMATO)}\nExplicación: {ejemplo['frase']}\n"
            f"Ahora tú:\n{ce.entrada_explicacion(f['ajustes'], FORMATO)}\nExplicación:")

print("Entrada de ejemplo:", ce.entrada_explicacion(test[0]["ajustes"], FORMATO))
vlm.eval()
with ce.Cronometro() as c_A3:
    salidas_A3, m_A3 = medir(vlm, prompt_A3, "A3")
print(f"A3: {c_A3.s:.0f} s para {len(test)} frases")

Entrada de ejemplo: Cambios: exposicion: sube bastante; temperatura: sube bastante; tinte: sin cambio; contraste: sube bastante; saturacion: sube bastante


A3 {'pct_ajustes_sentido_correcto': 40.7, 'pct_ajustes_sentido_contrario': 39.5, 'pct_frases_con_alucinacion': 58.3, 'pct_frases_espanol_longitud_ok': 0.0, 'pct_frases_perfectas': 0.0, 's_por_frase': 4.51}
A3: 270 s para 60 frases


## 3. B3 · LoRA: aprender a redactar

La misma LoRA que en el notebook 03 (r = 16 en la atención del modelo de lenguaje). Como la tarea es solo texto, los datos de entrenamiento son **gratis**: fabricamos `N_FRASES` juegos de ajustes al azar y sus frases con las reglas.

Con `FORMATO = "facil"`, la entrada ya trae el sentido y la intensidad de cada ajuste («temperatura: sube bastante»): el modelo solo tiene que **redactar**. Con `FORMATO = "json"`, recibe el JSON tal cual y tiene que decidir él el sentido. En el estudio, esa variante salió mucho peor.

In [5]:
prompt_B3 = lambda f: f"{ce.entrada_explicacion(f['ajustes'], FORMATO)}\nExplica el estilo en español."

def ejemplo_entrenamiento(f):
    pre = ids_prompt(prompt_B3(f))
    resp = tok(" " + f["frase"] + "<end_of_utterance>", return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    ids = torch.cat([pre, resp]); etiquetas = ids.clone(); etiquetas[:len(pre)] = -100
    return ids, etiquetas

def lote(ejs):
    n = max(len(i) for i, _ in ejs)
    ids = torch.full((len(ejs), n), tok.pad_token_id); mascara = torch.zeros((len(ejs), n), dtype=torch.long)
    etiquetas = torch.full((len(ejs), n), -100)
    for j, (i, e) in enumerate(ejs):
        ids[j, :len(i)], mascara[j, :len(i)], etiquetas[j, :len(i)] = i, 1, e
    return {"input_ids": ids, "attention_mask": mascara, "labels": etiquetas}

ej_train, ej_val = [ejemplo_entrenamiento(f) for f in train], [ejemplo_entrenamiento(f) for f in val]
print("Tokens de un ejemplo:", len(ej_train[0][0]))

for p in vlm.parameters():
    p.requires_grad = False
modelo = get_peft_model(vlm, LoraConfig(r=RANGO_LORA, lora_alpha=2 * RANGO_LORA, lora_dropout=0.05,
                                        target_modules=r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj)"))
modelo.print_trainable_parameters()
optimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=TASA_APRENDIZAJE)

def perdida_val():
    modelo.eval()
    with torch.no_grad():
        return float(np.mean([modelo(**lote(ej_val[i:i + 8])).loss.item() for i in range(0, len(ej_val), 8)]))

historia = {"epoca": [0], "perdida_val": [perdida_val()], "perdida_train": [float("nan")]}
with ce.Cronometro() as c_train:
    for epoca in range(1, EPOCAS + 1):
        modelo.train(); random.shuffle(ej_train); perdidas = []
        for i in range(0, len(ej_train), TAM_LOTE):
            perdida = modelo(**lote(ej_train[i:i + TAM_LOTE])).loss
            perdida.backward(); optimizador.step(); optimizador.zero_grad(); perdidas.append(perdida.item())
        historia["epoca"].append(epoca); historia["perdida_train"].append(float(np.mean(perdidas))); historia["perdida_val"].append(perdida_val())
        print(f"Época {epoca}: pérdida media de entrenamiento {historia['perdida_train'][-1]:.3f}, validación {historia['perdida_val'][-1]:.3f}")
print(f"Entrenamiento: {c_train.s:.0f} s")

Tokens de un ejemplo: 137


trainable params: 1,843,200 || all params: 258,328,128 || trainable%: 0.7135


Época 1: pérdida media de entrenamiento 0.309, validación 0.061


Época 2: pérdida media de entrenamiento 0.046, validación 0.032
Entrenamiento: 906 s


In [6]:
modelo.eval()
salidas_B3, m_B3 = medir(modelo, prompt_B3, "B3")
carpeta_adaptador = trabajo / "adaptador_explicacion"
modelo.save_pretrained(carpeta_adaptador)
m_B3.update(s_entrenamiento=c_train.s, adaptador_mb=ce.tamano_mb(carpeta_adaptador / "adapter_model.safetensors"),
            epocas=EPOCAS, n_train=len(train))
memoria = ce.memoria_mb()
print("Adaptador:", m_B3["adaptador_mb"], "MB | memoria pico del proceso:", memoria, "MB")

B3 {'pct_ajustes_sentido_correcto': 100.0, 'pct_ajustes_sentido_contrario': 0.0, 'pct_frases_con_alucinacion': 18.3, 'pct_frases_espanol_longitud_ok': 100.0, 'pct_frases_perfectas': 81.7, 's_por_frase': 4.11}


Adaptador: 7.06 MB | memoria pico del proceso: 2886.7 MB


## 4. Comparación y ejemplos reales lado a lado

In [7]:
objetivo = ce.resumir_frases([ce.evaluar_frase(f["frase"], f["ajustes"]) for f in test])
filas = {"objetivo (reglas)": objetivo, "A3 one-shot": m_A3, "B3 LoRA": m_B3}
claves = ["pct_ajustes_sentido_correcto", "pct_ajustes_sentido_contrario", "pct_frases_con_alucinacion",
          "pct_frases_espanol_longitud_ok", "pct_frases_perfectas"]
print(f"{'':18s}" + "".join(f"{c.replace('pct_', '')[:22]:>24s}" for c in claves) + f"{'s/frase':>9s}")
for nombre, m in filas.items():
    print(f"{nombre:18s}" + "".join(f"{m[c]:23.1f}%" for c in claves) + (f"{m['s_por_frase']:9.2f}" if "s_por_frase" in m else ""))

for f, a, b in list(zip(test, salidas_A3, salidas_B3))[:6]:
    print("\nAJUSTES :", ce.a_json(f["ajustes"]))
    print("OBJETIVO:", f["frase"])
    print("A3      :", a.replace("\n", " ")[:300])
    print("B3      :", b.replace("\n", " ")[:300])

metricas = {"A3": m_A3, "B3": m_B3, "objetivo": objetivo, "s_A3_total": c_A3.s, "memoria_pico_mb": memoria,
            "modelo": MODELO, "n_test": len(test), "hilos": hilos, "historia_B3": historia}
(resultados / "metricas_explicacion.json").write_text(json.dumps(metricas, indent=1, ensure_ascii=False), encoding="utf-8")
(resultados / "explicaciones.jsonl").write_text("\n".join(json.dumps({"id": f["id"], "objetivo": f["frase"], "A3": a, "B3": b}, ensure_ascii=False)
                                                          for f, a, b in zip(test, salidas_A3, salidas_B3)), encoding="utf-8")

mlflow = ce.mlflow_local(trabajo)
with mlflow.start_run(run_name="A3_explicacion_one_shot"):
    mlflow.log_params({"version": "A3", "modelo": MODELO, "n_test": len(test), "hilos": hilos})
    mlflow.log_metrics(m_A3)
with mlflow.start_run(run_name="B3_explicacion_lora"):
    mlflow.log_params({"version": "B3", "modelo": MODELO, "rango_lora": RANGO_LORA, "epocas": EPOCAS, "tam_lote": TAM_LOTE,
                       "tasa_aprendizaje": TASA_APRENDIZAJE, "n_train": len(train), "n_test": len(test), "hilos": hilos})
    for e, v in zip(historia["epoca"], historia["perdida_val"]):
        mlflow.log_metric("perdida_val", v, step=e)
    mlflow.log_metrics({k: v for k, v in m_B3.items() if isinstance(v, float)})
    mlflow.log_artifacts(str(carpeta_adaptador), artifact_path="adaptador_explicacion")
print("\nRuns en MLflow:", sorted(r.info.run_name for r in mlflow.search_runs(experiment_names=["pico-llm-estilo"], output_format="list")))

                    ajustes_sentido_correc  ajustes_sentido_contra  frases_con_alucinacion  frases_espanol_longitu        frases_perfectas  s/frase
objetivo (reglas)                   100.0%                    0.0%                    0.0%                  100.0%                  100.0%
A3 one-shot                          40.7%                   39.5%                   58.3%                    0.0%                    0.0%     4.51
B3 LoRA                             100.0%                    0.0%                   18.3%                  100.0%                   81.7%     4.11

AJUSTES : {"exposicion": 0.45, "temperatura": 0.5, "tinte": 0.03, "contraste": 1.35, "saturacion": 1.69}
OBJETIVO: Para copiar este estilo, se sube bastante la exposición, se sube bastante la temperatura, se sube bastante el contraste y se sube bastante la saturación.
A3      : La referencia es más luminosa, más magenta, con menos contraste y más saturada: se sube bastante la exposición, se lleva el tinte bastant


Runs en MLflow: ['A3_explicacion_one_shot', 'A_zero_shot', 'B3_explicacion_lora', 'B_lora', 'C_color_ridge', 'C_mobilenet_ridge', 'D_reinhard', 'comparativa']


## 5. Conclusiones (ejecución de referencia: 500 JSON fabricados de train, formato «facil», 2 épocas, 60 de test)

1. **A3 (one-shot) no sirve**: mezcla el ejemplo con la tarea y sigue escribiendo el prompt («Ejemplo: Cambios: …»), así que ninguna frase tiene una longitud razonable.
2. **B3 (LoRA) acierta el sentido del 100 % de los ajustes y no invierte ninguno**, y el 81,7 % de las frases son perfectas. El fallo que queda es **inventar un adjetivo**, sobre todo «más oscura», en la primera mitad de la frase, cuando ese ajuste es casi nulo: un 18,3 % de frases con alucinación.
3. **Qué cambió respecto al primer intento** (210 JSON en el formato «json»: 60,5 % de sentido correcto): dos cosas a la vez.
   - Más datos, que aquí son gratis porque se fabrican.
   - Una entrada más fácil: el sentido y la intensidad ya vienen calculados por las reglas, y el modelo **solo redacta**.

   Es lo que mejor hace un LLM: convertir información estructurada en texto. Decidir el sentido a partir de números lo hacen mejor las reglas.
4. **Lección para producción.** Divide el trabajo: las reglas (o C_color) deciden *qué* decir y el LLM decide *cómo* decirlo. Y mide siempre con una métrica automática contra la verdad. En una ejecución anterior con la misma configuración (6 hilos y la CPU compartida) salió un 10 % de alucinación; aquí sale un 18,3 %. Pequeñas diferencias de ejecución mueven esta cifra, así que no hay que fiarse de una sola corrida.